In [85]:
import pandas as pd
df = pd.read_excel(r'D:\PYTHON\Machine Learning\Data_set\Crop_Yield_Messy_Dataset_500.xlsx')
df = df.drop_duplicates()
df.columns = df.columns.str.strip().str.lower()

In [87]:
import numpy as np
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].mask(df[numeric_cols] < 0, np.nan)

In [81]:
df['state'] = df['state'].str.strip().str.lower()
df['state'] = df['state'].replace({
    'wb' : 'west bengal',
    'westbengal' : 'west bengal'
})

df['district'] = df['district'].str.strip().str.lower()

df['crop'] = df['crop'].str.strip().str.lower()

df['season'] = df['season'].str.strip().str.lower()

In [10]:
from sklearn.model_selection import train_test_split

X = df.drop('yield_ton_per_hectare', axis=1)
y = df['yield_ton_per_hectare']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [83]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

num_cols = X_train.select_dtypes(include='number').columns
X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = imputer.transform(X_test[num_cols])

In [ ]:
from sklearn.impute import SimpleImputer

cat_imputer = SimpleImputer(strategy='most_frequent')

categorical_cols = X_train.select_dtypes(include='object').columns
X_train[categorical_cols] = cat_imputer.fit_transform(X_train[categorical_cols])
X_test[categorical_cols] = cat_imputer.transform(X_test[categorical_cols])

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

X_train['irrigation'] = le.fit_transform(X_train['irrigation'])
X_test['irrigation'] = le.transform(X_test['irrigation'])

In [72]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

cols = ['state', 'district', 'crop', 'season', 'soil_type']

X_train_encoded = encoder.fit_transform(X_train[cols])
X_test_encoded = encoder.transform(X_test[cols])

In [ ]:
X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(cols),
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(cols),
    index=X_test.index
)

In [ ]:
X_train = X_train.drop(cols, axis=1)
X_test = X_test.drop(cols, axis=1)

X_train = pd.concat([X_train, X_train_encoded], axis=1)
X_test = pd.concat([X_test, X_test_encoded], axis=1)

TypeError: cannot concatenate object of type '<class 'numpy.ndarray'>'; only Series and DataFrame objs are valid